To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

Read our **[Qwen3 Guide](https://docs.unsloth.ai/basics/qwen3-how-to-run-and-fine-tune)** and check out our new **[Dynamic 2.0](https://docs.unsloth.ai/basics/unsloth-dynamic-2.0-ggufs)** quants which outperforms other quantization methods!

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "beomi/Llama-3-Open-Ko-8B-Instruct-preview",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.6: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

beomi/Llama-3-Open-Ko-8B-Instruct-preview does not have a padding token! Will use pad_token = <|reserved_special_token_250|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.5.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [61]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [60]:
drive.flush_and_unmount()

In [62]:
!ls -la /content/drive/MyDrive/Temp

total 3266
-rw------- 1 root root 3344184 May 20 18:20 finetune7.dat


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [66]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, get_chat_template

# 데이터셋 로드
dataset = load_dataset("json", data_files="/content/drive/MyDrive/Temp/finetune7.dat", split="train")

# 표준화 (필요시)
dataset = standardize_sharegpt(dataset)

# 데이터셋 구조 확인
print("데이터셋 샘플:", dataset[0])

# 토크나이저 설정 (실제 모델에 맞게 변경하세요)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("beomi/Llama-3-Open-Ko-8B-Instruct-preview")

# 채팅 템플릿 설정
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    map_eos_token=True,
)

# 데이터 포맷팅 함수 - 데이터셋 구조에 맞게 수정
def formatting_prompts_func(example):
    # 'conversations' 필드 사용
    messages = example['conversations']

    try:
        # apply_chat_template은 대화 목록을 직접 받아야 함
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}
    except Exception as e:
        print(f"오류 발생: {e}")
        print(f"문제가 발생한 messages: {messages}")
        return {"text": ""}

# 하나씩 처리 (batched=False)
processed_dataset = dataset.map(formatting_prompts_func)

# 결과 확인
print("\n처리된 데이터셋:")
print(processed_dataset[0])

데이터셋 샘플: {'conversations': [{'content': '너는 오늘 기분이 어때?', 'role': 'user'}, {'content': '저는 할머니랑 대화할 수 있어서 기분이 참 좋아요!', 'role': 'assistant'}, {'content': '나도 너랑 얘기하니까 기분이 한결 좋아진다.', 'role': 'user'}]}
Model does not have a padding token! Will use pad_token = <|reserved_special_token_250|>.


Map:   0%|          | 0/5581 [00:00<?, ? examples/s]


처리된 데이터셋:
{'conversations': [{'content': '너는 오늘 기분이 어때?', 'role': 'user'}, {'content': '저는 할머니랑 대화할 수 있어서 기분이 참 좋아요!', 'role': 'assistant'}, {'content': '나도 너랑 얘기하니까 기분이 한결 좋아진다.', 'role': 'user'}], 'text': '<|im_start|>user\n너는 오늘 기분이 어때?<|im_end|>\n<|im_start|>assistant\n저는 할머니랑 대화할 수 있어서 기분이 참 좋아요!<|im_end|>\n<|im_start|>user\n나도 너랑 얘기하니까 기분이 한결 좋아진다.<|im_end|>\n'}


In [67]:
print(dataset[0])

{'conversations': [{'content': '너는 오늘 기분이 어때?', 'role': 'user'}, {'content': '저는 할머니랑 대화할 수 있어서 기분이 참 좋아요!', 'role': 'assistant'}, {'content': '나도 너랑 얘기하니까 기분이 한결 좋아진다.', 'role': 'user'}]}


<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [74]:
from datasets import Dataset
import torch

# 데이터셋에서 텍스트 추출 및 전처리
texts = []
for example in dataset:
    try:
        if "conversations" in example:
            messages = example["conversations"]
            # 각 메시지를 간단한 형식으로 변환
            text = tokenizer.apply_chat_template(messages, tokenize=False)
            texts.append(text)
        else:
            print("conversations 필드가 없음:", example.keys())
    except Exception as e:
        print(f"변환 중 오류: {e}")
        print("문제가 발생한 예제:", example)

print(f"추출된 텍스트 수: {len(texts)}")
if texts:
    print(f"첫 번째 텍스트 샘플: {texts[0][:100]}...")

# 각 텍스트를 직접 토크나이즈
tokenized_texts = []
for text in texts:
    try:
        # 토큰화 및 텐서 변환
        tokens = tokenizer(text, truncation=True, max_length=max_seq_length)
        tokenized_texts.append({
            "input_ids": tokens["input_ids"],
            "attention_mask": tokens["attention_mask"]
        })
    except Exception as e:
        print(f"토크나이즈 중 오류: {e}")

print(f"토크나이즈된 텍스트 수: {len(tokenized_texts)}")
if tokenized_texts:
    print(f"첫 번째 토큰화 결과 길이: {len(tokenized_texts[0]['input_ids'])}")

# 새로운 데이터셋 생성
tokenized_dataset = Dataset.from_list(tokenized_texts)
print("토크나이즈된 데이터셋 크기:", len(tokenized_dataset))

# 학습기 초기화 - 이미 토크나이즈된 데이터셋 사용
from transformers import Trainer  # 일반 Trainer 사용

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
)

# 일반 Trainer 사용 (SFTTrainer 대신)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print("학습기 초기화 성공!")

추출된 텍스트 수: 5581
첫 번째 텍스트 샘플: <|im_start|>user
너는 오늘 기분이 어때?<|im_end|>
<|im_start|>assistant
저는 할머니랑 대화할 수 있어서 기분이 참 좋아요!<|im_end|...
토크나이즈된 텍스트 수: 5581
첫 번째 토큰화 결과 길이: 74
토크나이즈된 데이터셋 크기: 5581
학습기 초기화 성공!


<ipython-input-74-86595a5c70a9>:65: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [76]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.117 GB of memory reserved.


In [80]:
# 패딩과 자르기를 적용하여 데이터셋 준비
processed_data = []

for example in dataset:
    try:
        if "conversations" in example:
            messages = example["conversations"]
            text = tokenizer.apply_chat_template(messages, tokenize=False)

            # 토큰화 - padding과 truncation 적용
            encoded = tokenizer(
                text,
                truncation=True,
                padding='max_length',  # 최대 길이로 패딩
                max_length=max_seq_length,
                return_tensors=None  # 텐서로 변환하지 않고 파이썬 리스트로 유지
            )

            # input_ids를 labels로도 사용
            processed_data.append({
                "input_ids": encoded["input_ids"],
                "attention_mask": encoded["attention_mask"],
                "labels": encoded["input_ids"].copy()
            })
        else:
            continue
    except Exception as e:
        print(f"처리 중 오류: {e}")
        continue

# 새로운 데이터셋 생성
from datasets import Dataset
labeled_dataset = Dataset.from_list(processed_data)
print("레이블이 추가된 데이터셋 크기:", len(labeled_dataset))

# 데이터셋 검증 - 모든 시퀀스 길이가 동일한지 확인
if len(labeled_dataset) > 0:
    first_len = len(labeled_dataset[0]["input_ids"])
    print(f"첫 번째 항목 길이: {first_len}")

    # 몇 개의 항목을 샘플링하여 길이 확인
    import random
    sample_indices = random.sample(range(len(labeled_dataset)), min(5, len(labeled_dataset)))
    for idx in sample_indices:
        print(f"샘플 {idx} 길이: {len(labeled_dataset[idx]['input_ids'])}")

# Trainer 설정
from transformers import Trainer, TrainingArguments
from unsloth import is_bfloat16_supported

# 학습 인자 설정
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
)

# 데이터 콜레이터 정의 - 배치 처리 시 동일한 길이를 보장
from transformers import DefaultDataCollator
data_collator = DefaultDataCollator()

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=labeled_dataset,
    data_collator=data_collator,  # 데이터 콜레이터 추가
)

# 훈련 시작
trainer_stats = trainer.train()

레이블이 추가된 데이터셋 크기: 5581
첫 번째 항목 길이: 2048
샘플 465 길이: 2048
샘플 3434 길이: 2048
샘플 1087 길이: 2048
샘플 2710 길이: 2048
샘플 1760 길이: 2048


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,581 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,17.825200
2,17.953200
3,17.965800
4,15.103400
5,8.806400
6,3.419900
7,0.427100
8,0.164100
9,0.161300
10,0.135800


In [81]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2505.6675 seconds used for training.
41.76 minutes used for training.
Peak reserved memory = 8.262 GB.
Peak reserved memory for training = 1.145 GB.
Peak reserved memory % of max memory = 56.048 %.
Peak reserved memory for training % of max memory = 7.767 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



In [85]:
# Alpaca 프롬프트 형식 정의
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{0}

### Input:
{1}

### Response:
{2}"""

# 모델 추론 모드로 전환
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# 입력 생성
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "노인에게 친절하게 대한다",  # instruction - {0}
            "날이 춥나, 허리가 아프다",  # input - {1}
            "",  # output - {2} - leave this blank for generation!
        )
    ],
    return_tensors="pt"
).to("cuda")

# 생성
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
result = tokenizer.batch_decode(outputs)

# 결과 출력
print(result[0])

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
노인에게 친절하게 대한다

### Input:
날이 춥나, 허리가 아프다

### Response:
할머니, 따뜻한 차 한 잔 드릴까요?<|im_end|>


 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [107]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "노인에게 친절하게 대한다",  # instruction - {0}
        "글쎄다. 사람은 다 외롭지.",  # input - {1}
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
노인에게 친절하게 대한다

### Input:
글쎄다. 사람은 다 외롭지.

### Response:
그분께도 말이라도 걸어드려야겠어요.<|im_end|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [11]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [12]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is a famous tall tower in Paris?

### Input:


### Response:
One of the most famous and iconic landmarks in Paris is the Eiffel Tower. Standing at a height of 324 meters (1,063 feet), it is one of the tallest structures in the city and is a symbol of Paris and France. It was built for the 1889 World's Fair and was originally intended to be a temporary structure. However, it quickly became a popular tourist attraction and has been a prominent feature of the Paris skyline ever since. The Eiffel Tower is made up of three levels, with the first two levels being observation decks and the top level being a restaurant. Visitors can take the elevator or climb


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [13]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [14]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [15]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
